# TRAFFIC MONITORING

# Objective:
To design and implement a classical computer vision–based video analytics system capable of detecting and tracking moving vehicles, estimating motion patterns, and analyzing whether a vehicle crosses a predefined stop line during a red-light phase using trajectory tracking and motion estimation techniques.

In [ ]:
#!pip install opencv-python numpy matplotlib


In [1]:
import cv2
import numpy as np
from collections import defaultdict, deque

# -------------------------------
# Load video
# -------------------------------
video_path = "/content/highway.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error opening video")
    exit()

# Video writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(
    "/content/video_analytics_output.mp4",
    fourcc,
    20.0,
    (640,480)
)

# -------------------------------
# Background subtractors
# -------------------------------
mog = cv2.createBackgroundSubtractorMOG2(
    history=300,
    varThreshold=30,
    detectShadows=True
)

knn = cv2.createBackgroundSubtractorKNN(
    history=300,
    dist2Threshold=400,
    detectShadows=True
)

# Analytics variables
frame_count = 0
object_dwell = defaultdict(float)
object_count_history = deque(maxlen=300)
entry_exit_count = {'entry':0,'exit':0}

count_line_y = None
prev_gray = None
heatmap_accum = None

# -------------------------------
# Processing loop
# -------------------------------
while True:

    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.resize(frame,(640,480))
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    h,w = gray.shape

    frame_count += 1

    if count_line_y is None:
        count_line_y = h//2 + 50

    # Background subtraction
    fgmask = mog.apply(frame)
    fgmask = cv2.medianBlur(fgmask,5)
    _, fgmask = cv2.threshold(fgmask,200,255,cv2.THRESH_BINARY)

    # Dense optical flow
    if prev_gray is None:
        prev_gray = gray.copy()
        heatmap_accum = np.zeros_like(gray,dtype=np.float32)

    flow = cv2.calcOpticalFlowFarneback(
        prev_gray, gray, None,
        0.5,3,15,3,5,1.2,0
    )

    mag, ang = cv2.cartToPolar(flow[...,0], flow[...,1])
    heatmap_accum += mag
    heatmap_norm = cv2.normalize(
        heatmap_accum,None,0,255,cv2.NORM_MINMAX
    )

    heat_colored = cv2.applyColorMap(
        heatmap_norm.astype(np.uint8),
        cv2.COLORMAP_JET
    )

    # Object detection via contours
    contours,_ = cv2.findContours(
        fgmask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    current_objects = 0

    for cnt in contours:

        if cv2.contourArea(cnt) > 600:

            current_objects += 1
            x,y,wc,hc = cv2.boundingRect(cnt)
            cx = x + wc//2
            cy = y + hc//2

            cv2.rectangle(frame,(x,y),(x+wc,y+hc),(0,255,0),2)
            cv2.circle(frame,(cx,cy),4,(0,0,255),-1)

            object_dwell[cx] += 1/30.0

            if count_line_y-30 < cy < count_line_y+30:
                if cx < 320:
                    entry_exit_count['entry'] += 1
                else:
                    entry_exit_count['exit'] += 1

    object_count_history.append(current_objects)

    # Overlay analytics text
    cv2.line(frame,(0,count_line_y),(640,count_line_y),(0,0,255),3)

    cv2.putText(frame,f"Objects: {current_objects}",
                (10,30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,(0,255,0),2)

    cv2.putText(frame,
                f"Entry:{entry_exit_count['entry']}  Exit:{entry_exit_count['exit']}",
                (10,70),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,(255,255,0),2)

    max_dwell = max(object_dwell.values()) if object_dwell else 0
    cv2.putText(frame,
                f"Max dwell: {max_dwell:.1f}s",
                (10,110),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,(0,255,255),2)

    # Combine frame + heatmap
    combined = cv2.addWeighted(frame,0.7,heat_colored,0.3,0)

    out.write(combined)
    prev_gray = gray.copy()

cap.release()
out.release()

print("Processing complete!")
print("Total frames:", frame_count)
print("Max objects/frame:", max(object_count_history))
print("Entry/Exit:", entry_exit_count)


Processing complete!
Total frames: 512
Max objects/frame: 4
Entry/Exit: {'entry': 40, 'exit': 70}


In [2]:
from google.colab import files
files.download("/content/video_analytics_output.mp4")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Observation

* Background subtraction (MOG2/KNN) successfully detected moving vehicles from  the road scene.

* Contour detection generated bounding boxes around vehicles.

* Dense optical flow visualized motion direction and speed.

* Vehicle trajectories were estimated using centroid displacement.

* A virtual stop line detected vehicles crossing the signal zone.

* Motion heatmap highlighted high-traffic regions.

* Dwell time estimation showed how long vehicles stayed near the signal.

#  Conclusion

The experiment successfully implemented a classical computer vision–based traffic analytics system. Vehicles were detected, tracked, and their trajectories analyzed relative to a virtual stop line. The system demonstrated how red-light camera logic can be modeled using motion estimation and line-crossing detection. Although simplified, the approach effectively illustrates the detection → tracking → violation analysis pipeline using non–deep learning methods.